# 03 — YOLOv8 Training
Sanity check with YOLOv8n (5 epochs), then full training with YOLOv8m (50 epochs).

In [ ]:
import shutil
import os

source_dir = '/kaggle/input/datasets/bdd100k-yolo-subset/bdd100k_project'
destination_dir = '/kaggle/working/bdd100k_project'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print(f"Files copied to {destination_dir} and ready to use!")
else:
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

In [ ]:
!pip install ultralytics --no-deps

In [ ]:
import sys
import os
import numpy as np

project_path = "/kaggle/working/bdd100k_project"
if project_path not in sys.path:
    sys.path.append(project_path)
    sys.path.append(os.path.join(project_path, 'src'))

from src.utils import seed_everything, log_environment

seed_everything()
log_environment()

In [ ]:
import os
import shutil
import re

DATA_CONFIG = "/kaggle/working/bdd100k_project/configs/yolov8_bdd100k.yaml"
PROJECT_DIR = "/kaggle/working/bdd100k_project/runs"

os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    translate=0.1,
    scale=0.5,
    degrees=0.0,
)

## Sanity Check — YOLOv8n (5 epochs)

In [ ]:
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data=DATA_CONFIG,
    epochs=5,
    imgsz=640,
    batch=16,
    name="yolov8n_bdd100k_sanity",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    **AUGMENTATION,
)

In [ ]:
print("Sanity check complete.")
print(f"Results saved to: {os.path.join(PROJECT_DIR, 'train/yolov8n_bdd100k_sanity')}")

import glob
sanity_dir = os.path.join(PROJECT_DIR, "train/yolov8n_bdd100k_sanity")
for f in sorted(glob.glob(os.path.join(sanity_dir, "*.png"))):
    print(f"  {os.path.basename(f)}")

## Full Training — YOLOv8m (50 epochs)

In [ ]:
model_m = YOLO("yolov8s.pt")

results_m = model_m.train(
    data=DATA_CONFIG,
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolov8m_bdd100k_v1",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    patience=15,
    save=True,
    plots=True,
    **AUGMENTATION,
)

## Save Outputs

In [ ]:
best_weights_src = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1/weights/best.pt")
results_csv_src = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1/results.csv")

if os.path.exists(best_weights_src):
    shutil.copy(best_weights_src, "/kaggle/working/yolov8m_bdd100k_best.pt")
    print(f"Best weights saved to /kaggle/working/yolov8m_bdd100k_best.pt")

if os.path.exists(results_csv_src):
    shutil.copy(results_csv_src, "/kaggle/working/yolov8m_results.csv")
    print(f"Results CSV saved to /kaggle/working/yolov8m_results.csv")

run_dir = os.path.join(PROJECT_DIR, "train/yolov8m_bdd100k_v1")
if os.path.exists(run_dir):
    print(f"\nFull run directory: {run_dir}")
    for f in sorted(os.listdir(run_dir)):
        print(f"  {f}")

In [ ]:
import pandas as pd

results_csv = "/kaggle/working/yolov8m_results.csv"
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"Training completed: {len(df)} epochs")
    print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
    print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
    display(df.tail())

In [ ]:
import shutil
from IPython.display import FileLink

folder_path = "/kaggle/working/bdd100k_project"

zip_path = "/kaggle/working/bdd100k_project.zip"

shutil.make_archive(zip_path.replace('.zip',''), 'zip', folder_path)

# Display a download link
FileLink(zip_path)